## 1. Import thư viện

Cài đặt các thư viện cần thiết cho web scraping và xử lý dữ liệu.

In [ ]:
import csv
import re
import time
from typing import List, Dict, Optional
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

## 2. Cấu hình URL và Headers

Thiết lập URL gốc, URL danh sách phòng trọ HCM, và headers để mô phỏng trình duyệt.

In [ ]:
BASE_URL = "https://tromoi.com"
HCMC_LIST_URL = BASE_URL + "/phong-tro/ho-chi-minh"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
}

# Năm cần lọc
TARGET_YEAR = 2025

## 3. Hàm tiện ích: Làm sạch text

Loại bỏ khoảng trắng thừa và chuẩn hóa chuỗi.

In [ ]:
def clean_text(text: Optional[str]) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()

## 4. Trích xuất giá tiền

Lấy giá từ text dạng **'3.500.000 ₫/tháng'** hoặc **'3,5 triệu/tháng'**.  
Nếu không tìm thấy, trả về `"Liên hệ"` hoặc chuỗi rỗng.

In [ ]:
def extract_price(text: str) -> str:
    """
    Lấy giá dạng '3.500.000 ₫/tháng' hoặc '3,5 triệu/tháng'.
    Không tìm thấy thì trả về "" hoặc 'Liên hệ' nếu có chữ đó.
    """
    text = clean_text(text)
    if not text:
        return ""

    m = re.search(r"([\d\.,]+ ?₫ ?/tháng)", text)
    if m:
        return m.group(1)

    m2 = re.search(r"([\d\.,]+ ?triệu/tháng)", text, flags=re.IGNORECASE)
    if m2:
        return m2.group(1)

    if "liên hệ" in text.lower():
        return "Liên hệ"

    return ""

## 5. Chuyển đổi giá sang số VND

Nhận vào chuỗi giá từ `extract_price()` và trả về **số nguyên VND** (dạng chuỗi) hoặc rỗng.

**Ví dụ:**  
- `'3,5 triệu/tháng'` → `'3500000'`  
- `'3.500.000 ₫/tháng'` → `'3500000'`  
- `'Liên hệ'` → `''`

In [ ]:
def convert_price_to_number(price_text: str) -> str:
    """
    Nhận vào chuỗi do extract_price() trả về:
      - '3,5 triệu/tháng'   -> '3500000'
      - '3.500.000 ₫/tháng' -> '3500000'
      - 'Liên hệ' hoặc ''   -> ''
    Trả về chuỗi chỉ chứa số (VND) hoặc ''.
    """
    price_text = clean_text(price_text)
    if not price_text:
        return ""

    # 'Liên hệ' => rỗng
    if "liên hệ" in price_text.lower():
        return ""

    # Nếu có "triệu/tháng"
    if "triệu" in price_text.lower():
        m = re.search(r"([\d\.,]+)", price_text)
        if m:
            num_str = m.group(1).strip()
            # '3,5' hoặc '3.5'
            num_str = num_str.replace(".", "").replace(",", ".")
            try:
                val = float(num_str)
                vnd = int(round(val * 1_000_000))
                return str(vnd)
            except ValueError:
                return ""

    # Nếu có '₫/tháng' hoặc 'đ/tháng'
    if "₫" in price_text or "đ/tháng" in price_text.lower():
        m = re.search(r"([\d\.\,]+)", price_text)
        if m:
            digits = re.sub(r"\D", "", m.group(1))
            return digits

    return ""

## 6. Trích xuất diện tích

Tìm diện tích trong text toàn trang. Bắt các dạng: `'30m²'`, `'30 m2'`, `'Khoảng 30m²'`, ...

In [ ]:
def extract_area_from_text(text: str) -> str:
    """
    Tìm diện tích trong text toàn trang.
    Bắt các dạng: '30m²', '30 m2', 'Khoảng 30m²', ...
    """
    text = clean_text(text)
    m = re.search(r"(?:Khoảng\s*)?(\d+(?:,\d+)?)\s*m[²2]", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).replace(",", ".") + "m²"
    return ""

## 7. Trích xuất mô tả chi tiết

Ưu tiên các khối `<div>` hoặc `<section>` có class đặc trưng cho mô tả.  
Nếu không có, fallback sang heading **"Giới thiệu"** và gom các sibling bên dưới.

In [ ]:
def extract_description(soup: BeautifulSoup) -> str:
    """
    Cố gắng lấy phần mô tả chi tiết:
      1. Ưu tiên các khối div/section có class thường dùng cho mô tả.
      2. Nếu không có, fallback: tìm heading 'Giới thiệu' rồi gom các sibling bên dưới.
    """

    # 1) ưu tiên các khối mô tả "đặc trưng"
    candidates = soup.select(
        "div.detail-intro, "
        "div.room-intro, "
        "div.description, "
        "div#description, "
        "section.intro, "
        "section.description, "
        "article .content, "
        "div.content-detail"
    )
    for node in candidates:
        text = clean_text(node.get_text(" ", strip=True))
        if len(text) > 30:
            return text

    # 2) Fallback: heading "Giới thiệu"
    header = soup.find(
        lambda tag: tag.name in ["h2", "h3", "h4"]
        and "Giới thiệu" in tag.get_text()
    )
    if not header:
        return ""

    lines = []
    for sib in header.next_siblings:
        if getattr(sib, "name", None) in ["h1", "h2", "h3", "h4"]:
            break
        if hasattr(sib, "get_text"):
            t = clean_text(sib.get_text(" ", strip=True))
        else:
            t = clean_text(str(sib))
        if t:
            lines.append(t)

    return "\n".join(lines)

## 8. Lọc theo năm đăng

Tìm chuỗi **"Ngày đăng: dd-mm-yyyy"** và trả về năm (int).  
Nếu không tìm thấy, trả về `None`.

In [ ]:
def extract_posted_year(soup: BeautifulSoup) -> Optional[int]:
    """
    Tìm 'Ngày đăng: dd-mm-yyyy' và trả về năm (int).
    Không tìm thấy thì trả về None.
    """
    node = soup.find(
        string=lambda t: isinstance(t, str) and "Ngày đăng" in t
    )
    if not node:
        return None

    if hasattr(node, "parent"):
        text = clean_text(node.parent.get_text(" ", strip=True))
    else:
        text = clean_text(node)

    m = re.search(r"(\d{1,2}-\d{1,2}-\d{4})", text)
    if not m:
        return None

    date_str = m.group(1)  # vd: 20-10-2019
    parts = date_str.split("-")
    if len(parts) != 3:
        return None
    try:
        year = int(parts[2])
        return year
    except ValueError:
        return None

## 9. Tìm khối giá trong trang chi tiết

Tìm đoạn text chứa giá (`triệu/tháng` hoặc `₫/tháng`).  
Không phụ thuộc vào cụm "Giá chỉ từ" nữa.

In [ ]:
def find_price_block(soup: BeautifulSoup) -> str:
    """
    Tìm 1 đoạn text có chứa giá (triệu/tháng hoặc ₫/tháng, đ/tháng).
    Không phụ thuộc 'Giá chỉ từ' nữa.
    """
    for node in soup.find_all(["span", "div"]):
        txt = clean_text(node.get_text(" ", strip=True))
        low = txt.lower()
        if "triệu/tháng" in low or "triệu / tháng" in low:
            return txt
        if "₫/tháng" in txt or "đ/tháng" in low:
            return txt
    return ""

## 10. Crawl trang chi tiết

Truy cập URL chi tiết, lọc theo năm đăng, trích xuất thông tin và trả về dict.

**Bước xử lý:**
1. Lấy HTML và parse bằng BeautifulSoup
2. Kiểm tra năm đăng (chỉ lấy năm 2025)
3. Trích xuất: tiêu đề, địa chỉ, giá, diện tích, mô tả
4. Trả về dict hoặc `None` nếu không phù hợp

In [ ]:
def crawl_detail(url: str) -> Optional[Dict]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"[ERROR] Không load được chi tiết: {url} - {e}")
        return None

    soup = BeautifulSoup(resp.text, "lxml")

    # --- Lọc theo năm đăng ---
    year = extract_posted_year(soup)
    if year != TARGET_YEAR:
        print(f"[SKIP] {url} vì năm đăng {year}")
        return None

    # Title
    title_tag = soup.find("h1")
    title = clean_text(title_tag.get_text()) if title_tag else ""

    # Địa chỉ: lấy đoạn text ngay sau h1 (địa chỉ nguyên gốc)
    address = ""
    if title_tag:
        for sib in title_tag.next_siblings:
            if hasattr(sib, "get_text"):
                t = clean_text(sib.get_text(" ", strip=True))
                if t:
                    address = t
                    break

    # Text toàn trang
    page_text = soup.get_text(" ", strip=True)

    # ===== Giá =====
    raw_price_block = find_price_block(soup)   # text như '3,5 triệu/tháng' hoặc '3.500.000 đ/tháng'
    price_text = extract_price(raw_price_block)
    price_number = convert_price_to_number(price_text)  # chỉ còn số hoặc ""

    # Diện tích
    area_text = extract_area_from_text(page_text)

    # Mô tả
    description = extract_description(soup)

    record = {
        "url": url,
        "title": title,
        "price": price_number,   # CHỈ số VND hoặc ""
        "area": area_text,
        "address": address,
        "description": description,
        "source": "tromoi.com",
    }
    return record

## 11. Lấy danh sách links từ list page

Truy cập trang danh sách phòng trọ (list page) và thu thập tất cả links chi tiết.

**Lọc:**
- Chỉ lấy link dạng `/phong-tro/<slug-tin>`
- Loại bỏ link category `/phong-tro/ho-chi-minh`
- Loại bỏ các link trùng lặp

In [ ]:
def get_listing_links(page: int) -> List[str]:
    """
    Lấy link chi tiết từ trang list phòng trọ TP.HCM.
    Loại bỏ link category /phong-tro/ho-chi-minh, chỉ giữ /phong-tro/<slug-tin>.
    """
    if page == 1:
        u = HCMC_LIST_URL
    else:
        u = f"{HCMC_LIST_URL}?page={page}"

    try:
        resp = requests.get(u, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"[ERROR] Không load được list page {page}: {e}")
        return []

    soup = BeautifulSoup(resp.text, "lxml")
    links: List[str] = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        # Chỉ quan tâm link chứa /phong-tro/
        if "/phong-tro/" not in href:
            continue

        # Chuẩn hóa sang dạng tương đối dựa trên phần sau '/phong-tro/'
        idx = href.find("/phong-tro/")
        tail = href[idx + len("/phong-tro/"):]      # phần sau /phong-tro/
        tail = tail.split("?", 1)[0].strip("/")    # bỏ query & dấu '/'

        # Loại: rỗng, 'ho-chi-minh', hoặc không có dấu '-'
        if not tail:
            continue
        if tail == "ho-chi-minh":
            continue
        if "-" not in tail:
            continue

        full = urljoin(BASE_URL, "/phong-tro/" + tail)
        links.append(full)

    # Loại trùng
    links = list(dict.fromkeys(links))
    print(f"[INFO] Trang {page}: tìm được {len(links)} link tin.")
    return links

## 12. Crawl toàn bộ danh sách (nhiều trang)

Lặp qua các trang từ `start_page` đến `end_page` (bao gồm cả hai đầu).  
Thu thập tất cả records và trả về list dict.

**Tham số:**
- `start_page`: trang bắt đầu (VD: 1)
- `end_page`: trang kết thúc (VD: 30)
- `delay`: thời gian chờ giữa các request (giây)

In [ ]:
def crawl_all(start_page: int, end_page: int, delay: float = 1.5) -> List[Dict]:
    """
    Crawl từ trang start_page đến end_page (bao gồm).
    Ví dụ: start_page=3, end_page=10 -> cào trang 3..10
    """
    all_records: List[Dict] = []

    for p in range(start_page, end_page + 1):
        print(f"\n========== LIST PAGE {p} ==========")
        list_links = get_listing_links(p)

        for link in list_links:
            print(f"[DETAIL] Crawling: {link}")
            rec = crawl_detail(link)
            if rec:
                all_records.append(rec)
                # Debug vài record đầu
                if len(all_records) <= 2:
                    print("    → sample price =", rec["price"],
                          "| area =", rec["area"])
            else:
                print("    → [SKIP] Không lưu record cho link này")
            time.sleep(delay)

        time.sleep(delay)

    return all_records

## 13. Lưu kết quả vào CSV

Ghi danh sách records vào file CSV với encoding UTF-8-sig (hỗ trợ tiếng Việt tốt).

**Cột dữ liệu:**
- url
- title
- price (chỉ số VND)
- area
- address
- description
- source

In [ ]:
def save_csv(records: List[Dict], filename: str = "tromoi_hcm_updated.csv"):
    fieldnames = ["url", "title", "price", "area", "address",
                  "description", "source"]
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in records:
            writer.writerow(r)
    print(f"[DONE] Đã lưu {len(records)} dòng vào {filename}")

## 14. Chạy scraper

Thực thi toàn bộ pipeline scraping từ trang 1 đến trang 30 và lưu kết quả.

In [ ]:
# Crawl từ trang 1 đến 30 với delay 0.1s
data = crawl_all(start_page=1, end_page=30, delay=0.1)

# Lưu vào file CSV
save_csv(data, "data_tromoi.csv")

print(f"\n✅ Hoàn tất! Tổng cộng thu thập được {len(data)} records.")